# NB16：企業案例 — Agentic 智能自動化系統設計

## 情境說明
台灣某中型製造業（年營收約 15 億 NTD）導入 AI Agent 自動化三大核心流程，目標是降低人力成本、加速決策速度並減少人為錯誤。

## 系統架構圖

```
┌─────────────────────────────────────────────────────────┐
│                   企業 AI 自動化平台                      │
│                                                         │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐  │
│  │  採購審批     │  │  供應商研究   │  │  月報生成    │  │
│  │  工作流       │  │  （並行）     │  │  自動化      │  │
│  └──────┬───────┘  └──────┬───────┘  └──────┬───────┘  │
│         │                 │                  │          │
│  ┌──────▼───────────────────────────────────▼───────┐  │
│  │              OpenAI GPT-4o-mini                   │  │
│  │         Tool Calling / Orchestration              │  │
│  └──────────────────────┬────────────────────────────┘  │
│                         │                               │
│  ┌──────────────────────▼────────────────────────────┐  │
│  │  Mock Tools: 廠商DB / 預算系統 / ERP / 政策知識庫  │  │
│  └───────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────┘
```

## 自動化決策矩陣

| 工作流 | 規則複雜度 | 資料量 | 人工時間/月 | 適合自動化？ |
|--------|-----------|--------|------------|-------------|
| 採購審批 | 中 | 低 | 40 小時 | ✅ 是 |
| 供應商研究 | 高 | 高 | 80 小時 | ✅ 是 |
| 月報生成 | 中 | 中 | 30 小時 | ✅ 是 |
| 合約談判 | 極高 | 低 | 20 小時 | ❌ 否（需人工） |
| 員工績效 | 高 | 中 | 15 小時 | ⚠️ 部分輔助 |

---
**學習目標：**
1. 設計多步驟 Agent 工作流
2. 實作 Human-in-the-Loop 機制
3. 使用 asyncio 實作並行 Agent 執行
4. 計算自動化 ROI

In [ ]:
# Part 0：環境設定
import os
import json
import asyncio
import time
import random
from datetime import datetime, timedelta
from typing import Any
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

def llm(system: str, user: str, max_tokens: int = 512) -> str:
    """共用 LLM 呼叫函數"""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        max_tokens=max_tokens,
        temperature=0.3
    )
    return resp.choices[0].message.content.strip()

print(f"✅ 環境就緒 | 模型：{MODEL} | 時間：{datetime.now().strftime('%Y-%m-%d %H:%M')}")

## Part 1：採購審批工作流

### 流程設計
```
採購申請 → 廠商審查 → 預算確認 → 政策審查 → 風險評估 → 核准建議
                                                        ↓
                                            金額 > 50萬 → 人工確認
```

### Human-in-the-Loop 觸發條件
- 採購金額超過 **500,000 NTD**
- 廠商信用評分低於 **60 分**
- 政策合規警告 **2 項以上**

In [ ]:
# Part 1：採購審批工具函數（Mock）

# Mock 廠商資料庫
VENDOR_DB = {
    "V001": {"name": "台灣精密零件股份有限公司", "credit_score": 85, "years": 12, "blacklist": False},
    "V002": {"name": "新興材料有限公司", "credit_score": 55, "years": 2, "blacklist": False},
    "V003": {"name": "信達供應鏈管理", "credit_score": 90, "years": 8, "blacklist": False},
}

# Mock 部門預算
BUDGET_DB = {
    "製造部": {"total": 2_000_000, "used": 1_200_000},
    "研發部": {"total": 1_500_000, "used": 800_000},
    "採購部": {"total": 3_000_000, "used": 1_800_000},
}

# Mock 政策規則庫（簡化 RAG）
POLICY_RULES = [
    "單筆採購超過 50 萬 NTD 需總經理核准",
    "新廠商首次合作金額不得超過 100 萬 NTD",
    "進口物料需附海關文件，交期確認後方可下單",
    "危險化學品採購需環安部門預先審查",
    "同一廠商單季採購上限為年度預算的 30%",
]

def check_vendor(vendor_id: str) -> dict:
    """查詢廠商信用與合規狀態"""
    if vendor_id not in VENDOR_DB:
        return {"status": "not_found", "message": f"廠商 {vendor_id} 不存在資料庫"}
    v = VENDOR_DB[vendor_id]
    return {
        "vendor_id": vendor_id,
        "name": v["name"],
        "credit_score": v["credit_score"],
        "cooperation_years": v["years"],
        "blacklisted": v["blacklist"],
        "risk_level": "低" if v["credit_score"] >= 75 else ("中" if v["credit_score"] >= 55 else "高")
    }

def check_budget(dept: str, amount: float) -> dict:
    """確認部門預算是否充足"""
    if dept not in BUDGET_DB:
        return {"status": "dept_not_found"}
    b = BUDGET_DB[dept]
    remaining = b["total"] - b["used"]
    return {
        "dept": dept,
        "requested": amount,
        "remaining_budget": remaining,
        "sufficient": remaining >= amount,
        "utilization_after": round((b["used"] + amount) / b["total"] * 100, 1)
    }

def check_policy(query: str) -> dict:
    """使用 LLM + 政策規則庫進行合規審查（簡化 RAG）"""
    rules_text = "\n".join([f"{i+1}. {r}" for i, r in enumerate(POLICY_RULES)])
    result = llm(
        system="你是企業採購合規審查員。根據提供的政策規則，判斷採購請求是否合規，列出適用規則和警告。用 JSON 格式回應：{\"compliant\": bool, \"applicable_rules\": [], \"warnings\": []}",
        user=f"政策規則：\n{rules_text}\n\n採購請求：{query}"
    )
    try:
        return json.loads(result.strip('`').replace('json\n',''))
    except:
        return {"compliant": True, "applicable_rules": [], "warnings": []}

def assess_risk(vendor_info: dict, amount: float) -> dict:
    """評估採購風險等級"""
    risk_score = 0
    factors = []
    if vendor_info.get("credit_score", 100) < 70:
        risk_score += 30
        factors.append("廠商信用評分偏低")
    if vendor_info.get("cooperation_years", 10) < 3:
        risk_score += 20
        factors.append("合作年資不足")
    if amount > 500_000:
        risk_score += 25
        factors.append("採購金額超過門檻")
    if amount > 1_000_000:
        risk_score += 25
        factors.append("採購金額極高")
    level = "低" if risk_score < 30 else ("中" if risk_score < 60 else "高")
    return {"risk_score": risk_score, "risk_level": level, "risk_factors": factors}

print("✅ 採購工具函數載入完成")
# 快速測試
print("廠商查詢測試：", check_vendor("V001"))

In [ ]:
# Part 1：採購審批 Agent

class ProcurementAgent:
    """採購審批 Agent：整合多工具並決策，支援 Human-in-the-Loop"""

    HUMAN_THRESHOLD = 500_000  # NTD

    def process(self, request: dict, human_approved: bool = False) -> dict:
        print(f"\n{'='*55}")
        print(f"📋 採購申請：{request['description']}")
        print(f"   廠商：{request['vendor_id']} | 部門：{request['dept']} | 金額：{request['amount']:,.0f} NTD")
        print(f"{'='*55}")

        # Step 1：廠商審查
        vendor = check_vendor(request["vendor_id"])
        print(f"[1/4] 廠商審查 → 信用評分：{vendor.get('credit_score')} | 風險：{vendor.get('risk_level')}")

        # Step 2：預算確認
        budget = check_budget(request["dept"], request["amount"])
        print(f"[2/4] 預算確認 → 充足：{budget['sufficient']} | 使用率將達：{budget.get('utilization_after')}%")

        # Step 3：政策審查
        policy_query = f"廠商：{vendor.get('name')}，採購品項：{request['description']}，金額：{request['amount']} NTD，部門：{request['dept']}"
        policy = check_policy(policy_query)
        print(f"[3/4] 政策審查 → 合規：{policy.get('compliant')} | 警告數：{len(policy.get('warnings', []))}")

        # Step 4：風險評估
        risk = assess_risk(vendor, request["amount"])
        print(f"[4/4] 風險評估 → 風險分數：{risk['risk_score']} | 等級：{risk['risk_level']}")

        # Human-in-the-Loop 判斷
        needs_human = request["amount"] > self.HUMAN_THRESHOLD or vendor.get("credit_score", 100) < 60

        if needs_human and not human_approved:
            print(f"\n⚠️  [人工確認Required] 金額超過門檻或廠商評分偏低")
            print(f"   → 已發送審批請求給採購主管，等待確認...")
            return {"status": "pending_human_approval", "reason": "需要人工審核", "request": request}

        if needs_human and human_approved:
            print(f"\n✅ [人工確認] 主管已核准，繼續自動流程")

        # LLM 最終建議
        summary = f"""
        廠商信用：{vendor.get('credit_score')}/100，合作年資：{vendor.get('cooperation_years')}年
        預算充足：{budget['sufficient']}，政策合規：{policy.get('compliant')}
        風險等級：{risk['risk_level']}，風險因素：{risk['risk_factors']}
        警告事項：{policy.get('warnings', [])}
        """
        recommendation = llm(
            system="你是企業採購審批系統。根據審查結果給出明確的審批建議（核准/有條件核准/拒絕）和理由，請用繁體中文回應，限 100 字內。",
            user=summary
        )

        approved = policy.get("compliant", True) and budget["sufficient"] and not vendor.get("blacklisted", False)
        print(f"\n🤖 AI 建議：{recommendation}")
        print(f"\n📌 最終結果：{'✅ 核准' if approved else '❌ 拒絕'}")

        return {
            "status": "approved" if approved else "rejected",
            "recommendation": recommendation,
            "vendor": vendor,
            "budget": budget,
            "risk": risk,
        }


agent = ProcurementAgent()

# 案例一：小額採購（自動核准）
req1 = {"vendor_id": "V001", "dept": "製造部", "amount": 280_000, "description": "精密軸承零件 200 個"}
result1 = agent.process(req1)

# 案例二：大額採購（需人工，模擬主管已核准）
req2 = {"vendor_id": "V002", "dept": "研發部", "amount": 750_000, "description": "新型複合材料測試批次"}
result2 = agent.process(req2, human_approved=True)

## Part 2：供應商研究（並行執行）

### 並行 vs 序列比較
```
序列執行：供應商A(3s) → 供應商B(3s) → 供應商C(3s) = 9 秒
並行執行：供應商A ┐
          供應商B ├─ asyncio.gather → 同時完成 ≈ 3 秒
          供應商C ┘
```

並行執行節省 **66%** 的等待時間，適合 I/O 密集型任務（API 呼叫、資料庫查詢）。

In [ ]:
# Part 2：供應商研究 Agent（並行執行）
import asyncio

# Mock 供應商資料
SUPPLIERS = [
    {"id": "S001", "name": "台積精密", "country": "台灣", "category": "金屬零件"},
    {"id": "S002", "name": "越南製造聯盟", "country": "越南", "category": "塑膠模組"},
    {"id": "S003", "name": "德國工業夥伴 GmbH", "country": "德國", "category": "精密儀器"},
]

def mock_web_search(supplier_name: str) -> dict:
    """模擬網路搜尋（含隨機延遲）"""
    time.sleep(random.uniform(0.1, 0.3))  # 模擬網路延遲
    profiles = {
        "台積精密": {"founded": 1998, "employees": 320, "certifications": ["ISO 9001", "IATF 16949"], "recent_news": "2024Q3 獲台積電供應商認證"},
        "越南製造聯盟": {"founded": 2015, "employees": 850, "certifications": ["ISO 9001"], "recent_news": "擴廠計畫，產能預計增加 40%"},
        "德國工業夥伴 GmbH": {"founded": 1975, "employees": 1200, "certifications": ["ISO 9001", "CE", "DIN"], "recent_news": "榮獲歐洲製造業卓越獎"},
    }
    return profiles.get(supplier_name, {"founded": "未知", "employees": "未知"})

def mock_credit_check(supplier_id: str) -> dict:
    """模擬信用查詢"""
    time.sleep(random.uniform(0.1, 0.2))
    scores = {"S001": 88, "S002": 72, "S003": 95}
    payment = {"S001": "準時", "S002": "偶有延遲", "S003": "準時"}
    return {"credit_score": scores.get(supplier_id, 70), "payment_history": payment.get(supplier_id, "未知")}

async def research_supplier_async(supplier: dict) -> dict:
    """非同步研究單一供應商"""
    loop = asyncio.get_event_loop()
    # 將同步函數包裝為非同步
    web_data = await loop.run_in_executor(None, mock_web_search, supplier["name"])
    credit_data = await loop.run_in_executor(None, mock_credit_check, supplier["id"])

    # LLM 生成供應商摘要
    summary_prompt = f"""
    供應商：{supplier['name']}（{supplier['country']}），類別：{supplier['category']}
    基本資料：{web_data}
    信用資料：{credit_data}
    """
    summary = await loop.run_in_executor(
        None, llm,
        "你是供應商評估專家，用繁體中文生成 2-3 句供應商優勢摘要。",
        summary_prompt
    )

    return {
        "supplier": supplier,
        "web_data": web_data,
        "credit": credit_data,
        "summary": summary
    }

async def parallel_research():
    """並行研究所有供應商並比較時間"""
    print("🔄 開始並行供應商研究...")
    t_start = time.time()
    results = await asyncio.gather(*[research_supplier_async(s) for s in SUPPLIERS])
    t_parallel = time.time() - t_start
    print(f"✅ 並行完成！耗時：{t_parallel:.2f} 秒（研究了 {len(results)} 家供應商）")
    return results

# 執行並行研究
research_results = asyncio.run(parallel_research())

for r in research_results:
    s = r["supplier"]
    print(f"\n📊 {s['name']}（{s['country']}）")
    print(f"   信用評分：{r['credit']['credit_score']} | 付款：{r['credit']['payment_history']}")
    print(f"   摘要：{r['summary'][:80]}...")

In [ ]:
# Part 2：比較 Agent — 生成結構化比較表

def comparison_agent(results: list) -> str:
    """綜合研究結果，生成供應商比較報告"""
    data_text = ""
    for r in results:
        s = r["supplier"]
        w = r["web_data"]
        c = r["credit"]
        data_text += f"""
        供應商：{s['name']}（{s['country']}）
        類別：{s['category']}，成立：{w.get('founded')}年，員工：{w.get('employees')}人
        認證：{w.get('certifications')}，信用：{c['credit_score']}分，付款：{c['payment_history']}
        近況：{w.get('recent_news')}
        """

    report = llm(
        system="你是採購策略顧問。根據供應商資料，用繁體中文生成一份 Markdown 格式的比較分析報告，包含：比較表格、優缺點分析、推薦排名（含理由）。",
        user=data_text,
        max_tokens=800
    )
    return report

comparison_report = comparison_agent(research_results)
print(comparison_report)

## Part 3：月報自動生成

### 流程架構
```
KPI 資料 → AnalysisAgent → InsightAgent → ReportDraftAgent → 完整月報
（ERP）      （趨勢/異常）   （5 大洞察）    （Markdown 組裝）
```

傳統月報需 **2-3 天**人工彙整；自動化後縮短至 **15 分鐘**。

In [ ]:
# Part 3：Mock KPI 資料 + AnalysisAgent

# Mock 月度 KPI 資料
KPI_DATA = {
    "period": "2024年11月",
    "sales": {
        "revenue": 125_000_000,  # NTD
        "prev_month": 118_000_000,
        "target": 120_000_000,
        "yoy_growth": 8.3,  # %
        "top_products": ["精密軸承A型", "液壓模組B系列", "感測器整合套件"]
    },
    "quality": {
        "defect_rate": 0.8,  # %（目標 <1%）
        "prev_month": 1.2,
        "customer_complaints": 3,
        "rework_cost": 380_000  # NTD
    },
    "delivery": {
        "otd_rate": 94.2,  # On-Time Delivery %（目標 >95%）
        "prev_month": 96.1,
        "avg_lead_time_days": 12,
        "delayed_orders": 8
    },
    "inventory": {
        "turnover": 6.8,  # 次/年（目標 >7）
        "excess_value": 2_100_000,  # NTD
        "stockout_incidents": 2,
        "raw_material_days": 25  # 天
    },
    "cost": {
        "total_opex": 98_000_000,
        "labor_ratio": 42.1,  # %
        "energy_cost": 3_200_000,
        "material_cost_ratio": 38.5  # %
    }
}

def analysis_agent(kpi: dict) -> str:
    """分析 KPI 趨勢與異常"""
    kpi_text = json.dumps(kpi, ensure_ascii=False, indent=2)
    return llm(
        system="你是製造業績效分析師。分析 KPI 資料，用繁體中文列出：(1)正向趨勢 (2)異常警示 (3)需關注指標，各 2-3 點，語氣專業簡潔。",
        user=f"KPI 資料：\n{kpi_text}",
        max_tokens=400
    )

analysis = analysis_agent(KPI_DATA)
print("📊 KPI 分析結果：")
print(analysis)

In [ ]:
# Part 3：InsightAgent + ReportDraftAgent

def insight_agent(kpi: dict, analysis: str) -> str:
    """從分析結果提煉 5 大關鍵洞察"""
    return llm(
        system="你是資深營運顧問。根據 KPI 分析，用繁體中文提出 5 大可行洞察（每點含：洞察觀察、影響、建議行動），格式化輸出。",
        user=f"分析結果：\n{analysis}\n\nKPI 摘要：營收 {KPI_DATA['sales']['revenue']:,} NTD，缺陷率 {KPI_DATA['quality']['defect_rate']}%，準時交貨率 {KPI_DATA['delivery']['otd_rate']}%",
        max_tokens=600
    )

def report_draft_agent(kpi: dict, analysis: str, insights: str) -> str:
    """組裝完整月報"""
    period = kpi["period"]
    s = kpi["sales"]
    q = kpi["quality"]
    d = kpi["delivery"]
    inv = kpi["inventory"]

    # 結構化組裝（部分固定格式 + LLM 生成摘要）
    exec_summary = llm(
        system="你是企業 CEO 的幕僚長，用繁體中文寫一段 80 字的執行摘要，語氣正式，重點突出業績亮點與主要挑戰。",
        user=f"{analysis}"
    )

    report = f"""# {period} 製造部門月度績效報告

**生成時間：** {datetime.now().strftime('%Y-%m-%d %H:%M')} ｜ **版本：** 自動生成 v1.0

---

## 執行摘要
{exec_summary}

---

## 一、財務績效
| 指標 | 本月實績 | 上月實績 | 目標 | 達成率 |
|------|---------|---------|------|------|
| 營業收入 | {s['revenue']/1e6:.1f}M NTD | {s['prev_month']/1e6:.1f}M NTD | {s['target']/1e6:.1f}M NTD | {s['revenue']/s['target']*100:.1f}% |
| 年增率 | {s['yoy_growth']}% | - | 8% | {'✅' if s['yoy_growth']>=8 else '❌'} |

## 二、品質指標
| 指標 | 本月 | 上月 | 目標 | 狀態 |
|------|------|------|------|------|
| 缺陷率 | {q['defect_rate']}% | {q['prev_month']}% | <1% | {'✅ 達標' if q['defect_rate']<1 else '❌ 未達標'} |
| 客訴件數 | {q['customer_complaints']} | - | <5 | {'✅' if q['customer_complaints']<5 else '❌'} |

## 三、交貨績效
| 指標 | 本月 | 上月 | 目標 | 狀態 |
|------|------|------|------|------|
| 準時交貨率 | {d['otd_rate']}% | {d['prev_month']}% | >95% | {'✅' if d['otd_rate']>=95 else '⚠️ 略低'} |
| 延遲訂單 | {d['delayed_orders']} 筆 | - | <5 筆 | {'✅' if d['delayed_orders']<5 else '❌'} |

## 四、庫存健康
- 庫存周轉率：{inv['turnover']} 次/年（目標 >7，{'⚠️ 略低' if inv['turnover']<7 else '✅ 達標'}）
- 呆料價值：{inv['excess_value']/1e4:.0f} 萬 NTD
- 原料備料天數：{inv['raw_material_days']} 天

---

## 五、關鍵洞察與行動建議
{insights}

---
*本報告由 AI 自動生成，數據來源：ERP 系統 | 審核：製造部主管*
"""
    return report

# 執行 Agent 鏈
print("⚙️  生成洞察中...")
insights = insight_agent(KPI_DATA, analysis)
print("📝 組裝報告中...")
monthly_report = report_draft_agent(KPI_DATA, analysis, insights)

print(monthly_report)

## Part 4：異常偵測與告警

### 架構
```
即時感測器資料
      ↓
AnomalyDetector（閾值規則 + LLM 解讀）
      ↓
RootCauseAgent（歷史資料分析）
      ↓
AlertDispatcher（路由到正確團隊）
      ↓
   [品管部] [設備部] [採購部] [管理層]
```

In [ ]:
# Part 4：異常偵測 Agent

# 監控指標閾值
THRESHOLDS = {
    "machine_temp": {"max": 85, "unit": "°C", "team": "設備部"},
    "defect_rate": {"max": 1.5, "unit": "%", "team": "品管部"},
    "line_speed": {"min": 85, "unit": "件/時", "team": "製造部"},
    "material_stock": {"min": 500, "unit": "kg", "team": "採購部"},
    "energy_kwh": {"max": 2000, "unit": "kWh/班", "team": "設備部"},
}

# Mock 歷史異常資料庫
HISTORICAL_ANOMALIES = [
    {"date": "2024-08-15", "metric": "machine_temp", "value": 92, "root_cause": "冷卻水循環泵故障", "resolution": "更換泵浦，4小時恢復"},
    {"date": "2024-09-03", "metric": "defect_rate", "value": 2.1, "root_cause": "原材料批次品質不良", "resolution": "隔離批次，聯繫供應商換貨"},
    {"date": "2024-10-20", "metric": "machine_temp", "value": 88, "root_cause": "散熱片積塵", "resolution": "清潔散熱系統，2小時恢復"},
]

class AnomalyDetector:
    def detect(self, metrics: dict) -> list:
        """偵測指標異常"""
        anomalies = []
        for metric, value in metrics.items():
            thresh = THRESHOLDS.get(metric, {})
            is_anomaly = False
            direction = ""
            if "max" in thresh and value > thresh["max"]:
                is_anomaly = True
                direction = f"超過上限 {thresh['max']}{thresh['unit']}"
            if "min" in thresh and value < thresh["min"]:
                is_anomaly = True
                direction = f"低於下限 {thresh['min']}{thresh['unit']}"

            if is_anomaly:
                # LLM 解讀異常
                interpretation = llm(
                    system="你是工廠智慧監控系統，用繁體中文一句話說明此異常的潛在風險。",
                    user=f"指標 {metric} 目前值 {value}{thresh.get('unit','')}，{direction}。"
                )
                anomalies.append({
                    "metric": metric,
                    "value": value,
                    "threshold_info": direction,
                    "responsible_team": thresh.get("team", "管理層"),
                    "interpretation": interpretation,
                    "detected_at": datetime.now().strftime("%H:%M:%S")
                })
        return anomalies

class RootCauseAgent:
    def investigate(self, anomaly: dict) -> str:
        """根據歷史資料分析根本原因"""
        similar = [h for h in HISTORICAL_ANOMALIES if h["metric"] == anomaly["metric"]]
        hist_text = json.dumps(similar, ensure_ascii=False) if similar else "無歷史紀錄"
        return llm(
            system="你是工廠維護專家，根據歷史異常紀錄，用繁體中文推測最可能的根本原因（1-2句）。",
            user=f"當前異常：{anomaly['metric']} = {anomaly['value']}，{anomaly['threshold_info']}\n歷史紀錄：{hist_text}"
        )

class AlertDispatcher:
    def dispatch(self, anomaly: dict, root_cause: str) -> dict:
        """路由告警到正確團隊"""
        action = llm(
            system="你是工廠告警系統，用繁體中文建議 1 項立即行動和 1 項預防措施，各一句話。",
            user=f"異常：{anomaly['interpretation']}\n根本原因：{root_cause}"
        )
        return {
            "alert_id": f"ALT-{datetime.now().strftime('%H%M%S')}",
            "metric": anomaly["metric"],
            "value": anomaly["value"],
            "team": anomaly["responsible_team"],
            "root_cause": root_cause,
            "recommended_action": action,
            "priority": "HIGH" if anomaly["metric"] in ["machine_temp", "defect_rate"] else "MEDIUM"
        }

# 注入 2 個異常情境
CURRENT_METRICS = {
    "machine_temp": 91,      # ⚠️ 異常：超過 85°C
    "defect_rate": 0.7,      # ✅ 正常
    "line_speed": 88,        # ✅ 正常
    "material_stock": 320,   # ⚠️ 異常：低於 500kg
    "energy_kwh": 1850,      # ✅ 正常
}

detector = AnomalyDetector()
root_agent = RootCauseAgent()
dispatcher = AlertDispatcher()

print("🔍 開始監控指標...")
anomalies = detector.detect(CURRENT_METRICS)
print(f"\n⚠️  偵測到 {len(anomalies)} 個異常")

alerts = []
for anomaly in anomalies:
    print(f"\n{'─'*50}")
    print(f"📌 [{anomaly['detected_at']}] {anomaly['metric']} = {anomaly['value']} ({anomaly['threshold_info']})")
    print(f"   LLM 解讀：{anomaly['interpretation']}")
    root_cause = root_agent.investigate(anomaly)
    print(f"   根本原因：{root_cause}")
    alert = dispatcher.dispatch(anomaly, root_cause)
    alerts.append(alert)
    print(f"   ✉️  告警 {alert['alert_id']} → {alert['team']} [{alert['priority']}]")
    print(f"   建議行動：{alert['recommended_action'][:100]}...")

## Part 5：自動化 ROI 計算

### ROI 計算公式
```
年度節省 = 人力節省 + 錯誤降低成本
年度投資 = 開發成本 + 維運成本 + OpenAI API 費用
ROI (%) = (年度節省 - 年度投資) / 年度投資 × 100
回收期  = 年度投資 / 月度節省
```

In [ ]:
# Part 5：ROI 計算

ROI_DATA = [
    {
        "workflow": "採購審批工作流",
        "hours_saved_per_month": 40,
        "hourly_cost_ntd": 600,          # 採購員平均時薪
        "error_reduction_ntd_monthly": 50_000,  # 採購錯誤平均成本
        "dev_cost_ntd": 300_000,         # 一次性開發費
        "monthly_ops_ntd": 8_000,        # OpenAI API + 維運
    },
    {
        "workflow": "供應商研究自動化",
        "hours_saved_per_month": 80,
        "hourly_cost_ntd": 700,
        "error_reduction_ntd_monthly": 80_000,
        "dev_cost_ntd": 450_000,
        "monthly_ops_ntd": 12_000,
    },
    {
        "workflow": "月報自動生成",
        "hours_saved_per_month": 30,
        "hourly_cost_ntd": 800,
        "error_reduction_ntd_monthly": 20_000,
        "dev_cost_ntd": 200_000,
        "monthly_ops_ntd": 5_000,
    },
]

print("="*70)
print(f"{'工作流':<18} {'月省人力':>10} {'月省錯誤':>10} {'月總節省':>10} {'回收期':>8} {'年ROI':>8}")
print("="*70)

total_annual_saving = 0
total_dev_cost = 0

for r in ROI_DATA:
    monthly_labor = r["hours_saved_per_month"] * r["hourly_cost_ntd"]
    monthly_error = r["error_reduction_ntd_monthly"]
    monthly_saving = monthly_labor + monthly_error
    monthly_net = monthly_saving - r["monthly_ops_ntd"]
    annual_saving = monthly_net * 12
    annual_invest = r["dev_cost_ntd"] + r["monthly_ops_ntd"] * 12
    roi_pct = (annual_saving - r["dev_cost_ntd"]) / annual_invest * 100
    payback_months = r["dev_cost_ntd"] / monthly_net

    total_annual_saving += annual_saving
    total_dev_cost += r["dev_cost_ntd"]

    print(f"{r['workflow']:<18} {monthly_labor/1e4:>8.1f}萬 {monthly_error/1e4:>8.1f}萬 "
          f"{monthly_net/1e4:>8.1f}萬 {payback_months:>6.1f}月 {roi_pct:>7.0f}%")

print("="*70)
print(f"\n📊 三項工作流年度總節省：{total_annual_saving/1e6:.2f}M NTD")
print(f"💰 總開發投資：{total_dev_cost/1e4:.0f} 萬 NTD")
print(f"🎯 整體年化 ROI：約 {(total_annual_saving-total_dev_cost)/total_dev_cost*100:.0f}%")

# LLM 生成 ROI 建議
roi_summary = llm(
    system="你是企業數位轉型顧問，用繁體中文寫 3 句話評估以上自動化投資的商業價值和建議優先順序。",
    user=f"三項工作流自動化，年度節省 {total_annual_saving/1e6:.1f}M NTD，開發投資 {total_dev_cost/1e4:.0f}萬 NTD，年化 ROI 約 {(total_annual_saving-total_dev_cost)/total_dev_cost*100:.0f}%。"
)
print(f"\n🤖 顧問建議：\n{roi_summary}")

## Part 6：面試 Q&A

---

### Q1：如何判斷哪些工作流適合 Agentic 自動化？

**A：** 評估四個維度：
1. **規則明確性**：流程是否有清晰的判斷邏輯？高度依賴直覺或政治判斷的任務不適合。
2. **重複頻率**：每月執行次數越多，自動化 ROI 越高（一般建議 >20 次/月才划算）。
3. **資料可及性**：輸入資料是否可結構化存取（API、資料庫）？人工蒐集資料的流程自動化難度高。
4. **錯誤容忍度**：錯誤代價大的流程（如財務、安全）需要 Human-in-the-Loop，不能完全自動化。

實務上使用「自動化決策矩陣」對候選流程評分，優先實施分數最高且風險最低的項目。

---

### Q2：Agent 失敗時如何處理？

**A：** 設計多層容錯機制：
1. **重試機制（Retry）**：對暫時性錯誤（API 逾時、網路中斷）實施指數退避重試，最多 3 次。
2. **降級策略（Fallback）**：LLM 失敗時回退到規則引擎；規則引擎失敗時路由給人工處理。
3. **死信佇列（Dead Letter Queue）**：連續失敗的任務進入 DLQ，避免阻塞整體流程。
4. **告警通知**：失敗超過閾值時自動通知值班工程師，附上完整的執行追蹤日誌。
5. **幕等性設計（Idempotency）**：確保重試不會產生重複操作（如重複下單、重複扣款）。

---

### Q3：Human-in-the-Loop 應如何設計？

**A：** 三個設計原則：
1. **明確觸發條件**：事先定義「何時需要人工介入」的規則（如金額門檻、置信度低於 0.7、違規警告等），避免主觀判斷。
2. **最小化打擾**：只在 AI 真正無法決策時介入，並提供 AI 的分析結論和建議供人類參考，而非從零開始審查。
3. **審計追蹤**：記錄每次人工介入的決定和理由，作為改善 AI 模型的訓練資料，逐漸減少介入頻率。

---

### Q4：並行 Agent 的排程與協調如何實作？

**A：** 根據任務相依性選擇架構：
- **完全獨立任務**：使用 `asyncio.gather()` 或 `concurrent.futures.ThreadPoolExecutor` 並行執行，無需協調。
- **部分相依任務**：建立 DAG（有向無環圖）定義相依關係，使用 Celery、Airflow 或自訂調度器執行。
- **競爭結果（Race）**：使用 `asyncio.wait(return_when=FIRST_COMPLETED)` 取最快完成的結果。
- **資源限制**：使用 `asyncio.Semaphore` 限制同時執行的 Agent 數量，避免 API 速率限制（Rate Limit）。

並行化的關鍵是識別哪些步驟有資料相依（必須序列），哪些步驟獨立（可並行）。

---

### Q5：如何衡量 Agentic 自動化的 ROI？

**A：** 分三個層次量化：

**硬性效益（可直接計算）：**
- 人力節省 = 每月省下工時 × 平均時薪 × 12
- 錯誤成本降低 = 歷史平均錯誤成本 × 錯誤率改善比例

**軟性效益（需間接估算）：**
- 決策速度提升帶來的商機（如採購提前導致缺料損失減少）
- 員工滿意度提升（減少重複性工作）帶來的留才效益

**成本項目：**
- 開發成本（一次性）+ 維運成本（每月）+ LLM API 費用

建議在上線前 3 個月設定 A/B 對照組（部分仍人工處理），以實測數據驗證 ROI 假設，避免高估效益。